# 05 - Indici dell'errore, commutazioni dell'infusione e sensibilità ai parametri di tetto IOB e PLGS

**Obiettivo.** Rispondere alle richieste della co-relatrice dopo la revisione del repository:
1. aggiungere un indice dell'errore (IAE) e il tempo di rientro nel range;
2. quantificare le commutazioni e la variabilità dell'infusione introdotte dai vincoli;
3. fare una breve analisi di sensibilità sui parametri principali del tetto IOB e del PLGS.

**Scenario.** Pasto non annunciato alle 12:00 da 60 e 90 g su `adult#001` e `adult#003` (i due pazienti in cui il
controllore proposto ha effetti opposti), stesso seme del sensore, stesse condizioni iniziali e assestamento del notebook 02.
I moduli sono attivati separatamente (`PID+IOB`, `PID+PLGS`) e insieme (`PID+IOB+PLGS`) a parità di PID.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
import run_campaign as RC          # stesse funzioni, stessa cache e simulazioni in parallelo della campagna
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_columns", 40)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = RC.COL
print("Versioni delle librerie usate:"); display(U.library_versions())

Versioni delle librerie usate:


,Pacchetto,Versione
0,Python,3.13.16
1,simglucose,0.2.11
2,numpy,2.4.4
3,scipy,1.17.1
4,pandas,3.0.2
5,matplotlib,3.10.8


## 1. Configurazione
### 1.1 Scenario: condizioni iniziali, durata, passo, pasto, seme

In [2]:
PAZ = RC.SENS_PAZ; CARICHI = RC.SENS_CHO
for p in PAZ:
    print(p); display(U.scenario_table(p, "60 o 90", SEED))

adult#001


,Voce,Valore,Unita'
0,Paziente virtuale,adult#001,-
1,Peso corporeo BW,102.3,kg
2,Glicemia basale di equilibrio Gb,138.6,mg/dL
3,Fabbisogno basale nominale u_b,1.267,U/h
4,Sensore CGM,Dexcom (rumore di simglucose),-
5,Pompa di infusione,Insulet,-
6,Seme casuale del sensore,1,-
7,Inizio simulazione,00:00 del giorno precedente (24 h prima),ora del giorno
8,Assestamento escluso dalle metriche,giorno precedente + 00:00-08:00 (32 h),-
9,Finestra di analisi,08:00-24:00,ora del giorno


adult#003


,Voce,Valore,Unita'
0,Paziente virtuale,adult#003,-
1,Peso corporeo BW,81.6,kg
2,Glicemia basale di equilibrio Gb,147.1,mg/dL
3,Fabbisogno basale nominale u_b,1.425,U/h
4,Sensore CGM,Dexcom (rumore di simglucose),-
5,Pompa di infusione,Insulet,-
6,Seme casuale del sensore,1,-
7,Inizio simulazione,00:00 del giorno precedente (24 h prima),ora del giorno
8,Assestamento escluso dalle metriche,giorno precedente + 00:00-08:00 (32 h),-
9,Finestra di analisi,08:00-24:00,ora del giorno


### 1.2 Parametri e vincoli del controllore proposto (con unità di misura)

In [3]:
ub = U.initial_conditions("adult#001", SEED)["ub"]
display(U.controller_table(A.CFG["PID+IOB+PLGS"], ub))

,Parametro,Valore,Unita',Significato
0,target,120.0,mg/dL,Glicemia obiettivo del regolatore
1,kp,0.0001,U/min per mg/dL,Guadagno proporzionale
2,ki,0.0,U/min per (mg/dL*min),Guadagno integrale
3,kd,0.002,U/min per (mg/dL/min),Guadagno derivativo sulla pendenza CGM
4,n_slope,5,campioni,Campioni CGM usati per stimare la pendenza ai minimi quadrati
5,aw_frac,1.0,adimensionale,Anti-windup: Ki*I_max = aw_frac * basale
6,aw_frac_low,0.25,adimensionale,Anti-windup: Ki*I_min = -aw_frac_low * basale
7,freeze,True,-,Integrazione condizionata quando l'uscita è limitata
8,use_iob,True,-,Attiva il tetto sull'insulina attiva (IOB)
9,iob_alpha,0.75,adimensionale,Tetto IOB = iob_alpha * basale orario


### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `Vincolo` | - | vincolo attivo nel passo: nessuno, `iob`, `plgs`, `sat_alta`, `sat_bassa` |

### Indicatori aggiunti (calcolati in `aid_sim.metrics`, finestra 08:00-24:00)
| Indicatore | Unità | Definizione |
|---|---|---|
| `IAE` | mg/dL·h | integrale dell'errore assoluto rispetto al target: $\int |BG(t) - 120|\,dt$ |
| `IAE_post6h` | mg/dL·h | come IAE, nelle 6 ore dopo il pasto |
| `IAE_range` | mg/dL·h | integrale della distanza dall'intervallo 70-180 (zero dentro il range): penalizza solo le escursioni cliniche |
| `t_rientro` | h | tempo dal pasto al rientro definitivo in 70-180 mg/dL (0 se non esce, NaN se a mezzanotte è ancora fuori) |
| `n_commut` | - | numero di cambi del vincolo attivo (attivazioni e disattivazioni di tetto IOB, PLGS, saturazioni) |
| `n_sosp` | - | numero di sospensioni dell'infusione (passaggi da u > 0 a u = 0) |
| `TV_u` | U/h | variazione totale dell'infusione $\sum_k |u_k - u_{k-1}|$ |
| `std_u` | U/h | deviazione standard dell'infusione |

`n_commut` e `n_sosp` misurano le commutazioni introdotte dai vincoli; `TV_u` misura anche le variazioni continue
del PID dovute al rumore del sensore, quindi è diversa da zero anche senza vincoli.

## 2. Indicatori aggiunti per modulo

In [4]:
MOSTRA = ["TIR", "TIR_post6h", "TAR", "TBR", "TBR_tardivo", "picco", "nadir",
         "IAE", "IAE_post6h", "IAE_range", "t_rientro", "n_commut", "n_sosp", "TV_u", "std_u"]
NOMI = ["PID", "PID+IOB", "PID+PLGS", "PID+IOB+PLGS"]
keys = [RC.key(p, c, n) for p in PAZ for c in CARICHI for n in NOMI]
RC.run_all(keys)
T = pd.DataFrame([{"Paziente": k[0], "CHO [g]": int(k[1]), "Controllore": k[2],
                   **{m: round(RC.cache[k][1][m], 1) for m in MOSTRA}} for k in keys])
display(T)

,Paziente,CHO [g],Controllore,TIR,TIR_post6h,TAR,TBR,TBR_tardivo,picco,nadir,IAE,IAE_post6h,IAE_range,t_rientro,n_commut,n_sosp,TV_u,std_u
0,adult#001,60,PID,75.0,33.3,25.0,0.0,0.0,197.3,90.8,460.5,342.1,27.3,5.0,0,0,8.5,0.3
1,adult#001,60,PID+IOB,74.7,32.5,25.3,0.0,0.0,197.7,99.6,445.1,345.3,28.4,5.1,54,0,49.8,0.3
2,adult#001,60,PID+PLGS,75.3,34.2,24.7,0.0,0.0,196.1,90.9,454.2,338.2,24.9,5.0,4,2,15.3,0.4
3,adult#001,60,PID+IOB+PLGS,74.7,32.5,25.3,0.0,0.0,196.8,98.8,438.1,342.8,26.4,5.0,62,0,56.8,0.3
4,adult#001,90,PID,67.5,13.3,32.5,0.0,0.0,236.8,79.6,680.8,515.3,184.4,5.9,0,0,8.8,0.4
5,adult#001,90,PID+IOB,65.3,11.7,34.7,0.0,0.0,239.3,102.0,668.0,528.6,198.5,6.2,122,0,121.3,0.4
6,adult#001,90,PID+PLGS,67.8,14.2,32.2,0.0,0.0,235.4,81.7,671.6,510.8,180.0,5.9,10,5,27.4,0.7
7,adult#001,90,PID+IOB+PLGS,65.3,11.7,34.7,0.0,0.0,238.6,101.6,660.2,526.5,196.1,6.2,128,0,127.2,0.4
8,adult#003,60,PID,73.8,30.0,26.2,0.0,0.0,204.2,81.4,559.5,379.0,69.6,5.0,0,0,8.6,0.3
9,adult#003,60,PID+IOB,72.2,25.8,27.8,0.0,0.0,209.0,91.5,529.1,402.3,81.3,5.3,54,0,57.3,0.3


## 3. Commutazioni dell'infusione
Infusione comandata con il vincolo attivo evidenziato, pasto da 60 g.

In [5]:
VCOL = {"iob": "tab:purple", "plgs": "tab:green", "sat_alta": "tab:red", "sat_bassa": "tab:gray"}
fig, ax = plt.subplots(len(NOMI), 2, figsize=(14, 10), sharex=True)
for j, p in enumerate(PAZ):
    ubp = A.basal_rate(A.make_patient(p))
    for i, n in enumerate(NOMI):
        d = RC.cache[RC.key(p, 60, n)][0]; d = d[d.t_h >= A.WARMUP_H]; a = ax[i, j]
        a.plot(d.t_h, d.Insulin_Uh, color=COL[n], drawstyle="steps-post", lw=1)
        for v, c in VCOL.items():
            on = (d.Vincolo == v).values
            if on.any():
                a.fill_between(d.t_h, 0, 1, where=on, step="post", color=c, alpha=0.25,
                               transform=a.get_xaxis_transform(), label=f"vincolo: {v}")
        a.axhline(ubp * 60, color="gray", ls=":", lw=1); a.axvline(A.MEAL_CLOCK_H, color="k", ls=":")
        m = RC.cache[RC.key(p, 60, n)][1]
        a.set_title(f"{p} - {n}: {m['n_commut']} commutazioni, {m['n_sosp']} sospensioni", fontsize=9)
        a.set_ylabel("U/h"); a.grid(ls=":", alpha=0.5)
        if a.get_legend_handles_labels()[0]: a.legend(fontsize=7, loc="upper right")
    ax[-1, j].set_xlabel("Ora del giorno [h]")
plt.tight_layout(); plt.show()

## 4. Sensibilità ai parametri del tetto IOB e del PLGS
Un parametro alla volta sul controllore `PID+IOB+PLGS`, gli altri al valore nominale.

| Parametro | Unità | Valori provati | Nominale |
|---|---|---|---|
| `iob_alpha` (tetto IOB = α · basale orario) | - | 0.5, 0.75, 1, 1.5 | 0.75 |
| `tau_s` (costante di tempo dello stimatore IOB) | min | 30, 50, 70 | 50 |
| `t_pred` (orizzonte di predizione del PLGS) | min | 10, 20, 30 | 20 |
| `plgs_pred` (soglia sulla glicemia proiettata) | mg/dL | 80, 90, 100 | 90 |

In [6]:
sk = [RC.key(p, c, "PID+IOB+PLGS", **{par: v}) for par, vals in RC.SENS.items() for v in vals
      for p in PAZ for c in CARICHI]
RC.run_all(sk)
S = pd.DataFrame([{"Parametro": k[6][0][0], "Valore": k[6][0][1], "Paziente": k[0], "CHO [g]": int(k[1]),
                   **{m: round(RC.cache[k][1][m], 1) for m in ["TIR", "TAR", "TBR", "TBR_tardivo", "picco",
                                                              "IAE_range", "t_rientro", "n_commut", "TV_u"]}}
                  for k in sk])
for par in RC.SENS:
    print(par); display(S[S.Parametro == par].drop(columns="Parametro").reset_index(drop=True))

iob_alpha


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,0.50,adult#001,60,73.4,26.6,0.0,0.0,199.4,32.4,5.2,156,110.8
1,0.50,adult#001,90,63.4,36.6,0.0,0.0,242.3,213.8,6.5,171,144.5
2,0.50,adult#003,60,70.0,30.0,0.0,0.0,219.7,104.0,5.7,148,123.7
3,0.50,adult#003,90,62.5,37.5,0.0,0.0,278.3,351.5,6.6,161,156.3
4,0.75,adult#001,60,74.7,25.3,0.0,0.0,196.8,26.4,5.0,62,56.8
5,0.75,adult#001,90,65.3,34.7,0.0,0.0,238.6,196.1,6.2,128,127.2
6,0.75,adult#003,60,72.8,27.2,0.0,0.0,206.7,71.3,5.2,62,71.1
7,0.75,adult#003,90,65.9,34.1,0.0,0.0,260.9,286.8,6.0,110,136.5
8,1.00,adult#001,60,75.3,24.7,0.0,0.0,196.1,24.9,5.0,16,25.8
9,1.00,adult#001,90,66.9,33.1,0.0,0.0,236.4,185.9,6.0,64,75.4


tau_s


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,30.0,adult#001,60,75.3,24.7,0.0,0.0,196.1,24.9,5.0,4,15.3
1,30.0,adult#001,90,67.5,32.5,0.0,0.0,235.7,182.0,5.9,34,51.5
2,30.0,adult#003,60,74.7,25.3,0.0,0.0,201.6,58.6,5.0,12,30.9
3,30.0,adult#003,90,69.1,30.9,0.0,0.0,244.1,235.7,5.5,36,62.2
4,50.0,adult#001,60,74.7,25.3,0.0,0.0,196.8,26.4,5.0,62,56.8
5,50.0,adult#001,90,65.3,34.7,0.0,0.0,238.6,196.1,6.2,128,127.2
6,50.0,adult#003,60,72.8,27.2,0.0,0.0,206.7,71.3,5.2,62,71.1
7,50.0,adult#003,90,65.9,34.1,0.0,0.0,260.9,286.8,6.0,110,136.5
8,70.0,adult#001,60,73.8,26.2,0.0,0.0,198.3,29.7,5.2,142,104.3
9,70.0,adult#001,90,63.7,36.2,0.0,0.0,241.0,207.8,6.5,168,143.9


t_pred


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,10.0,adult#001,60,75.3,24.7,0.0,0.0,197.0,25.4,5.0,54,49.1
1,10.0,adult#001,90,65.3,34.7,0.0,0.0,238.5,194.4,6.2,120,119.4
2,10.0,adult#003,60,72.8,27.2,0.0,0.0,208.8,80.0,5.2,58,64.8
3,10.0,adult#003,90,65.6,34.4,0.0,0.0,262.7,296.1,6.1,104,127.9
4,20.0,adult#001,60,74.7,25.3,0.0,0.0,196.8,26.4,5.0,62,56.8
5,20.0,adult#001,90,65.3,34.7,0.0,0.0,238.6,196.1,6.2,128,127.2
6,20.0,adult#003,60,72.8,27.2,0.0,0.0,206.7,71.3,5.2,62,71.1
7,20.0,adult#003,90,65.9,34.1,0.0,0.0,260.9,286.8,6.0,110,136.5
8,30.0,adult#001,60,74.1,25.9,0.0,0.0,197.9,32.7,5.1,72,67.9
9,30.0,adult#001,90,65.0,35.0,0.0,0.0,239.8,204.8,6.2,133,133.8


plgs_pred


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,80.0,adult#001,60,75.0,25.0,0.0,0.0,197.1,25.9,5.0,56,51.1
1,80.0,adult#001,90,65.3,34.7,0.0,0.0,238.7,195.3,6.2,124,123.2
2,80.0,adult#003,60,72.8,27.2,0.0,0.0,208.7,79.5,5.2,56,61.2
3,80.0,adult#003,90,65.6,34.4,0.0,0.0,262.6,295.6,6.1,102,124.1
4,90.0,adult#001,60,74.7,25.3,0.0,0.0,196.8,26.4,5.0,62,56.8
5,90.0,adult#001,90,65.3,34.7,0.0,0.0,238.6,196.1,6.2,128,127.2
6,90.0,adult#003,60,72.8,27.2,0.0,0.0,206.7,71.3,5.2,62,71.1
7,90.0,adult#003,90,65.9,34.1,0.0,0.0,260.9,286.8,6.0,110,136.5
8,100.0,adult#001,60,74.1,25.9,0.0,0.0,197.7,33.7,5.0,80,79.5
9,100.0,adult#001,90,65.0,35.0,0.0,0.0,239.7,206.0,6.2,145,146.6


In [7]:
MET = [("TIR", "TIR [%]"), ("TBR", "TBR [%]"), ("picco", "Picco [mg/dL]"), ("n_commut", "Commutazioni [-]")]
fig, ax = plt.subplots(len(MET), len(RC.SENS), figsize=(16, 11), sharex="col")
for j, (par, vals) in enumerate(RC.SENS.items()):
    for p in PAZ:
        for c, ls in zip(CARICHI, ["-", "--"]):
            for i, (m, lab) in enumerate(MET):
                y = [RC.cache[RC.key(p, c, "PID+IOB+PLGS", **{par: v})][1][m] for v in vals]
                line, = ax[i, j].plot(vals, y, marker="o", ls=ls, label=f"{p}, {c} g")
                ax[i, j].axhline(RC.cache[RC.key(p, c, "PID")][1][m], color=line.get_color(), ls=ls, lw=0.8, alpha=0.4)
    for i, (m, lab) in enumerate(MET):
        ax[i, j].axvline(RC.NOMINALE[par], color="k", ls=":", lw=1); ax[i, j].grid(ls=":", alpha=0.5)
        if j == 0: ax[i, j].set_ylabel(lab)
    ax[-1, j].set_xlabel(par)
ax[0, 0].legend(fontsize=7)
fig.suptitle("Linea verticale: valore nominale; linee orizzontali sottili: PID senza vincoli")
plt.tight_layout(); plt.show()

## 5. Lettura dei risultati (calcolata dalle tabelle)

In [8]:
for p in PAZ:
    for c in CARICHI:
        r = {n: T[(T.Paziente == p) & (T["CHO [g]"] == c) & (T.Controllore == n)].iloc[0] for n in NOMI}
        print(f"{p} {c} g: rispetto al PID")
        for n in NOMI[1:]:
            print(f"   {n:13s} dTIR {r[n].TIR - r['PID'].TIR:+6.1f}  dTBR {r[n].TBR - r['PID'].TBR:+6.1f}  "
                  f"dpicco {r[n].picco - r['PID'].picco:+6.1f}  dIAE_range {r[n].IAE_range - r['PID'].IAE_range:+7.1f}  "
                  f"commutazioni {int(r[n].n_commut):3d}  sospensioni {int(r[n].n_sosp):3d}")
print()
for par, vals in RC.SENS.items():
    d = S[S.Parametro == par]
    rng = d.groupby(["Paziente", "CHO [g]"])[["TIR", "TBR", "picco"]].agg(lambda x: x.max() - x.min()).max()
    print(f"{par:10s} escursione massima al variare del parametro: TIR {rng.TIR:5.1f} punti, "
          f"TBR {rng.TBR:5.1f} punti, picco {rng.picco:5.1f} mg/dL")

adult#001 60 g: rispetto al PID
   PID+IOB       dTIR   -0.3  dTBR   +0.0  dpicco   +0.4  dIAE_range    +1.1  commutazioni  54  sospensioni   0
   PID+PLGS      dTIR   +0.3  dTBR   +0.0  dpicco   -1.2  dIAE_range    -2.4  commutazioni   4  sospensioni   2
   PID+IOB+PLGS  dTIR   -0.3  dTBR   +0.0  dpicco   -0.5  dIAE_range    -0.9  commutazioni  62  sospensioni   0
adult#001 90 g: rispetto al PID
   PID+IOB       dTIR   -2.2  dTBR   +0.0  dpicco   +2.5  dIAE_range   +14.1  commutazioni 122  sospensioni   0
   PID+PLGS      dTIR   +0.3  dTBR   +0.0  dpicco   -1.4  dIAE_range    -4.4  commutazioni  10  sospensioni   5
   PID+IOB+PLGS  dTIR   -2.2  dTBR   +0.0  dpicco   +1.8  dIAE_range   +11.7  commutazioni 128  sospensioni   0
adult#003 60 g: rispetto al PID
   PID+IOB       dTIR   -1.6  dTBR   +0.0  dpicco   +4.8  dIAE_range   +11.7  commutazioni  54  sospensioni   0
   PID+PLGS      dTIR   +0.9  dTBR   +0.0  dpicco   -2.6  dIAE_range   -11.0  commutazioni  12  sospensioni   6
   PID+I

**Note per l'interpretazione.**
- Il confronto `PID+IOB` / `PID+PLGS` / `PID+IOB+PLGS` separa il contributo dei due moduli a parità di PID.
- Il tetto IOB è un interruttore (sopra il tetto l'infusione torna di colpo al basale) e il PLGS sospende e riprende
  l'infusione: entrambi introducono commutazioni, contate in `n_commut` e `n_sosp`.
- La sensibilità è fatta un parametro alla volta: non esplora le interazioni tra parametri e non è un'ottimizzazione.